# 选修实验——Softmax 函数
在本实验中，我们将探索 softmax 函数。在解决多分类（Multiclass Classification）问题时，Softmax 回归和神经网络都会用到这个函数。

<center>  <img  src="./images/C2_W2_Softmax_Header.PNG" width="600" />  <center/>

  

In [1]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from IPython.display import display, Markdown, Latex
from sklearn.datasets import make_blobs
%matplotlib widget
from matplotlib.widgets import Slider
from lab_utils_common import dlc
from lab_utils_softmax import plt_softmax
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

> **注意**：通常在本课程中，notebook 遵循从 0 开始、到 N-1 结束的计数约定，即 $\sum_{i=0}^{N-1}$；而课程讲座从 1 开始、到 N 结束，即 $\sum_{i=1}^{N}$。这是因为代码迭代一般从 0 开始，而讲座中从 1 数到 N 能让公式更简洁。本 notebook 的公式比一般实验多，因此将打破这个惯例，采用从 1 数到 N。

## Softmax 函数
在 softmax 回归和带 Softmax 输出的神经网络中，都会生成 N 个输出，并从中选出一个作为预测类别。两种情况下，都是由一个线性函数生成向量 $\mathbf{z}$，再对其应用 softmax 函数。softmax 函数把 $\mathbf{z}$ 转换成概率分布，如下所述。应用 softmax 之后，每个输出都在 0 到 1 之间，且所有输出之和为 1，因此可以解释为概率。输入越大，对应的输出概率越大。
<center>  <img  src="./images/C2_W2_SoftmaxReg_NN.png" width="600" />  

softmax 函数可以写成：
$$a_j = \frac{e^{z_j}}{ \sum_{k=1}^{N}{e^{z_k} }} \tag{1}$$
输出 $\mathbf{a}$ 是长度为 N 的向量，所以对 softmax 回归，也可以写成：
\begin{align}
\mathbf{a}(x) =
\begin{bmatrix}
P(y = 1 | \mathbf{x}; \mathbf{w},b) \\
\vdots \\
P(y = N | \mathbf{x}; \mathbf{w},b)
\end{bmatrix}
=
\frac{1}{ \sum_{k=1}^{N}{e^{z_k} }}
\begin{bmatrix}
e^{z_1} \\
\vdots \\
e^{z_{N}} \\
\end{bmatrix} \tag{2}
\end{align}


这表明输出是一个概率向量：第一个元素是在给定输入 $\mathbf{x}$ 和参数 $\mathbf{w}$、$\mathbf{b}$ 的条件下，输入属于第一类别的概率。
让我们创建一个 NumPy 实现：

In [2]:
def my_softmax(z):
    ez = np.exp(z)              #element-wise exponenial
    sm = ez/np.sum(ez)
    return(sm)

在下面，用滑块改变 `z` 输入的值。

In [3]:
plt.close("all")
plt_softmax(my_softmax)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

在调整上面各个 z 值的时候，注意以下几点：
* softmax 分子中的指数会放大取值之间的微小差异
* 输出值之和为 1
* softmax 会覆盖所有输出——例如改变 `z0` 会改变 `a0` 到 `a3` 的所有值。与 ReLU 或 Sigmoid 等单输入单输出的激活函数对比一下

## 代价
<center> <img  src="./images/C2_W2_SoftMaxCost.png" width="400" />    <center/>

与 Softmax 关联的损失函数是交叉熵损失（cross-entropy loss）：
\begin{equation}
  L(\mathbf{a},y)=\begin{cases}
    -log(a_1), & \text{if $y=1$}.\\
        &\vdots\\
     -log(a_N), & \text{if $y=N$}
  \end{cases} \tag{3}
\end{equation}

其中 y 是该样本的目标类别，$\mathbf{a}$ 是 softmax 函数的输出。具体来说，$\mathbf{a}$ 中的值是总和为 1 的概率。
>**回忆一下：** 本课程中，Loss（损失）针对单个样本，而 Cost（代价）覆盖所有样本。
 
 
注意在上面的 (3) 中，只有与目标对应的那一行对损失有贡献，其他行都为零。为了写出代价公式，我们需要一个“指示函数”（indicator function）：当索引与目标一致时为 1，否则为 0。
    $$\mathbf{1}\{y == n\} = =\begin{cases}
    1, & \text{if $y==n$}.\\
    0, & \text{otherwise}.
  \end{cases}$$
于是代价为：
\begin{align}
J(\mathbf{w},b) = -\frac{1}{m} \left[ \sum_{i=1}^{m} \sum_{j=1}^{N}  1\left\{y^{(i)} == j\right\} \log \frac{e^{z^{(i)}_j}}{\sum_{k=1}^N e^{z^{(i)}_k} }\right] \tag{4}
\end{align}

其中 $m$ 是样本数量，$N$ 是输出数量。这就是所有损失的平均值。


## Tensorflow
本实验将讨论在 Tensorflow 中实现 softmax 交叉熵损失的两种方式：“直观”方式和“推荐”方式。前者最直接，后者在数值上更稳定。

让我们先创建一个数据集来训练多分类模型。

In [4]:
# 为本例创建数据集
centers = [[-5, 2], [-2, -2], [1, 2], [5, -2]]
X_train, y_train = make_blobs(n_samples=2000, centers=centers, cluster_std=1.0,random_state=30)

### *直观*的组织方式

下面的模型把 softmax 作为最后一个 Dense 层的激活函数来实现。
损失函数则在 `compile` 指令中单独指定。

损失函数是 `SparseCategoricalCrossentropy`，即上面 (3) 所描述的损失。在这个模型中，softmax 发生在最后一层，损失函数接收的是 softmax 的输出，即一个概率向量。

In [5]:
model = Sequential(
    [ 
        Dense(25, activation = 'relu'),
        Dense(15, activation = 'relu'),
        Dense(4, activation = 'softmax')    # < softmax activation here
    ]
)
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(0.001),
)

model.fit(
    X_train,y_train,
    epochs=10
)
        

Epoch 1/10
63/63 [==============================] - 0s 966us/step - loss: 0.8312
Epoch 2/10
63/63 [==============================] - 0s 1ms/step - loss: 0.3203
Epoch 3/10
63/63 [==============================] - 0s 1ms/step - loss: 0.1408
Epoch 4/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0847
Epoch 5/10
63/63 [==============================] - 0s 944us/step - loss: 0.0626
Epoch 6/10
63/63 [==============================] - 0s 974us/step - loss: 0.0515
Epoch 7/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0447
Epoch 8/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0402
Epoch 9/10
63/63 [==============================] - 0s 922us/step - loss: 0.0361
Epoch 10/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0332


由于 softmax 集成在输出层中，模型的输出是一个概率向量。

In [6]:
p_nonpreferred = model.predict(X_train)
print(p_nonpreferred [:2])
print("largest value", np.max(p_nonpreferred), "smallest value", np.min(p_nonpreferred))

[[3.39e-03 1.02e-02 9.73e-01 1.30e-02]
 [9.97e-01 3.46e-03 9.35e-06 9.02e-06]]
largest value 0.9999994 smallest value 4.1378247e-09


### 推荐方式 <img align="Right" src="./images/C2_W2_softmax_accurate.png"  style=" width:400px; padding: 10px 20px ; ">
回忆课程讲座的内容：如果在训练期间把 softmax 和损失合并处理，可以得到更稳定、更准确的结果。这里展示的“推荐”组织方式就能做到这一点。


在推荐的组织方式中，最后一层使用线性激活。由于历史原因，这种形式下的输出被称为 *logits*。损失函数多了一个参数：`from_logits = True`，它告诉损失函数应把 softmax 运算纳入损失计算中，从而可以进行优化实现。

In [7]:
preferred_model = Sequential(
    [ 
        Dense(25, activation = 'relu'),
        Dense(15, activation = 'relu'),
        Dense(4, activation = 'linear')   #<-- Note
    ]
)
preferred_model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),  #<-- Note
    optimizer=tf.keras.optimizers.Adam(0.001),
)

preferred_model.fit(
    X_train,y_train,
    epochs=10
)
        

Epoch 1/10
63/63 [==============================] - 0s 1ms/step - loss: 1.0936
Epoch 2/10
63/63 [==============================] - 0s 1ms/step - loss: 0.4933
Epoch 3/10
63/63 [==============================] - 0s 1ms/step - loss: 0.2809
Epoch 4/10
63/63 [==============================] - 0s 1ms/step - loss: 0.1506
Epoch 5/10
63/63 [==============================] - 0s 966us/step - loss: 0.0916
Epoch 6/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0694
Epoch 7/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0592
Epoch 8/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0527
Epoch 9/10
63/63 [==============================] - 0s 919us/step - loss: 0.0489
Epoch 10/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0457


#### 输出的处理
注意，在推荐方式的模型中，输出不再是概率，而是可以取从很大的负数到很大的正数。当进行需要概率的预测时，必须把输出再过一遍 softmax。
让我们看看推荐模型的输出：

In [8]:
p_preferred = preferred_model.predict(X_train)
print(f"two example output vectors:\n {p_preferred[:2]}")
print("largest value", np.max(p_preferred), "smallest value", np.min(p_preferred))

two example output vectors:
 [[-2.72 -4.45  2.9  -1.37]
 [ 6.42  1.32 -1.32 -6.74]]
largest value 12.410254 smallest value -13.030992


输出的预测值不是概率！
如果需要概率形式的输出，应该用 [softmax](https://www.tensorflow.org/api_docs/python/tf/nn/softmax) 处理输出。

In [9]:
sm_preferred = tf.nn.softmax(p_preferred).numpy()
print(f"two example output vectors:\n {sm_preferred[:2]}")
print("largest value", np.max(sm_preferred), "smallest value", np.min(sm_preferred))

two example output vectors:
 [[3.55e-03 6.34e-04 9.82e-01 1.38e-02]
 [9.94e-01 6.05e-03 4.35e-04 1.92e-06]]
largest value 0.9999995 smallest value 1.5196424e-11


如果只是要选出最可能的类别，则不需要 softmax——用 [np.argmax()](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html) 找出最大输出的索引即可。

In [10]:
for i in range(5):
    print( f"{p_preferred[i]}, category: {np.argmax(p_preferred[i])}")

[-2.72 -4.45  2.9  -1.37], category: 2
[ 6.42  1.32 -1.32 -6.74], category: 0
[ 4.64  1.5  -1.31 -5.3 ], category: 0
[-0.29  3.76 -3.11 -1.58], category: 1
[-0.64 -6.05  5.23 -5.2 ], category: 2


## SparseCategorialCrossentropy 还是 CategoricalCrossEntropy
Tensorflow 的目标值有两种可能的格式，选择哪个损失函数决定了期望的格式：
- SparseCategorialCrossentropy：期望目标是对应索引的整数。例如，如果有 10 个可能的目标值，y 在 0 到 9 之间。
- CategoricalCrossEntropy：期望样本的目标值是独热（one-hot）编码——目标索引处的值为 1，其余 N-1 个位置为 0。例如有 10 个可能目标值、目标为 2 时，表示为 [0,0,1,0,0,0,0,0,0,0]。


## 恭喜！
在本实验中你：
- 更加熟悉了 softmax 函数，及其在 softmax 回归和神经网络 softmax 激活中的应用
- 学会了 Tensorflow 中推荐的模型构建方式：
    - 最后一层不加激活（等同于线性激活）
    - SparseCategoricalCrossentropy 损失函数
    - 使用 from_logits=True
- 认识到 softmax 与 ReLU、Sigmoid 不同，它会同时影响多个输出